# PYTHON-13. Визуализация данных (Урок 5)
## Практическая работа на данных Krisha.kz, Churn и Game of Thrones

**Датасеты:**
- `krisha_astana_esil_nura_2026.csv` — 1-комнатные квартиры в Астане (Есиль и Нура, 750 объявлений)
- `churn.csv` — данные о клиентах банка (10 000 записей): возраст, баланс, отток
- `character-deaths.csv` — смерти персонажей «Игры престолов»

**Библиотеки:** `pandas`, `matplotlib`, `seaborn`, `plotly.express`

**Схема выбора типа графика** (из материалов урока):
> *Что вы хотите показать?* → Сравнение / Взаимосвязь / Структуру / Распределение / Динамику → конкретный тип диаграммы

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless backend для сохранения в файл
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Загрузка датасетов
krisha_df = pd.read_csv('krisha_astana_esil_nura_2026.csv')
krisha_df['Price_m2'] = (krisha_df['price'] / krisha_df['area']).round().astype(int)
krisha_df['date'] = pd.to_datetime(krisha_df['date'])
krisha_df['Quarter'] = krisha_df['date'].dt.quarter
krisha_df['Month'] = krisha_df['date'].dt.month
krisha_df['FloorType'] = krisha_df.apply(
    lambda r: 'первый' if r['floor'] == 1 else ('последний' if r['floor'] == r['floor_count'] else 'средний'), axis=1
)

churn_df = pd.read_csv('churn.csv')

got_df = pd.read_csv('character-deaths.csv')
got_df['Allegiances'] = got_df['Allegiances'].str.replace('House ', '', regex=False)

print('Krisha:', krisha_df.shape, '| Churn:', churn_df.shape, '| GoT:', got_df.shape)
print('Krisha cols:', list(krisha_df.columns))
print('Churn cols:', list(churn_df.columns))
print('GoT cols:', list(got_df.columns))

---
## 1. Быстрые графики через Pandas `.plot()`
*(по мотивам `Visualization_Pandas.ipynb`)*

Pandas позволяет строить базовые графики прямо из DataFrame с помощью метода `.plot()`.
Параметр `kind` задаёт тип: `'line'`, `'bar'`, `'barh'`, `'hist'`, `'box'`, `'pie'`, `'scatter'`.

In [ ]:
# Линейный график — динамика количества объявлений по месяцам и районам
monthly = krisha_df.groupby(['Month', 'district'])['price'].count().unstack()
monthly.plot(
    kind='line',
    figsize=(12, 4),
    title='Динамика числа объявлений по месяцам (Krisha.kz, 2026)',
    grid=True,
    marker='o',
    lw=2
)
plt.xlabel('Месяц')
plt.ylabel('Число объявлений')
plt.tight_layout()
plt.savefig('plot_01_line_monthly.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 1 сохранён: plot_01_line_monthly.png')

In [ ]:
# Гистограмма — распределение цен за м²
krisha_df['Price_m2'].plot(
    kind='hist',
    bins=40,
    figsize=(10, 4),
    title='Распределение цены за 1 м² (₸)',
    color='steelblue',
    grid=True
)
plt.xlabel('Цена за м² (₸)')
plt.ylabel('Число объявлений')
plt.tight_layout()
plt.savefig('plot_02_hist_price_m2.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 2 сохранён: plot_02_hist_price_m2.png')

In [ ]:
# Столбчатая диаграмма — средняя цена за м² по районам и типу дома
district_house = krisha_df.groupby(['district', 'house_type'])['Price_m2'].mean().unstack()
district_house.plot(
    kind='bar',
    figsize=(10, 5),
    title='Средняя цена за м² по районам и типу дома',
    colormap='Set2',
    grid=True,
    legend=True
)
plt.xlabel('Район')
plt.ylabel('Средняя цена за м² (₸)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('plot_03_bar_district_house.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 3 сохранён: plot_03_bar_district_house.png')

---
## 2. Matplotlib — объектно-ориентированный подход
*(по мотивам `Visualizaion_Matplotlib.ipynb`)*

Matplotlib работает через объекты `Figure` (холст) и `Axes` (система координат).
`fig.add_axes([left, bottom, width, height])` — координаты в долях от размера холста.

In [ ]:
# Figure с несколькими axes — 3 графика в ряд по данным Churn
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
fig.suptitle('Анализ оттока клиентов банка (Churn)', fontsize=16, fontweight='bold')

# 1. Распределение возраста
axes[0].hist(churn_df['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Распределение возраста')
axes[0].set_xlabel('Возраст')
axes[0].set_ylabel('Количество клиентов')
axes[0].grid(True, alpha=0.3)

# 2. Средний баланс по стране
country_balance = churn_df.groupby('Geography')['Balance'].mean()
axes[1].bar(country_balance.index, country_balance.values, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[1].set_title('Средний баланс по стране')
axes[1].set_xlabel('Страна')
axes[1].set_ylabel('Средний баланс ($)')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Отток по полу
gender_churn = churn_df.groupby('Gender')['Exited'].mean() * 100
axes[2].bar(gender_churn.index, gender_churn.values, color=['#E91E63', '#2196F3'])
axes[2].set_title('Доля оттока по полу (%)')
axes[2].set_xlabel('Пол')
axes[2].set_ylabel('Доля ушедших клиентов (%)')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(gender_churn.values):
    axes[2].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_04_matplotlib_churn.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 4 сохранён: plot_04_matplotlib_churn.png')

In [ ]:
# Вложенный (insert) график — главный + врезка
# Главный: цена vs площадь по районам Астаны
# Врезка: распределение Price_m2 по FloorType

fig = plt.figure(figsize=(12, 6))
main_ax = fig.add_axes([0, 0, 1, 1])

colors = {'Есильский р-н': '#2196F3', 'Нура р-н': '#FF5722'}
for district, group in krisha_df.groupby('district'):
    main_ax.scatter(group['area'], group['price'] / 1_000_000,
                    label=district, alpha=0.5, s=30, color=colors[district])

main_ax.set_title('Зависимость цены от площади квартиры (Астана, 2026)', fontsize=14)
main_ax.set_xlabel('Площадь (м²)')
main_ax.set_ylabel('Цена (млн ₸)')
main_ax.legend()
main_ax.grid(alpha=0.3)

# Врезка: boxplot по FloorType
insert_ax = fig.add_axes([0.63, 0.55, 0.35, 0.4])
floor_data = [krisha_df[krisha_df['FloorType'] == ft]['Price_m2'].values for ft in ['первый', 'средний', 'последний']]
insert_ax.boxplot(floor_data, tick_labels=['первый', 'средний', 'последний'], patch_artist=True,
                  boxprops=dict(facecolor='lightyellow'))
insert_ax.set_title('Price_m2 по этажу', fontsize=9)
insert_ax.set_ylabel('₸/м²', fontsize=8)
insert_ax.tick_params(labelsize=8)

plt.savefig('plot_05_matplotlib_scatter_insert.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 5 сохранён: plot_05_matplotlib_scatter_insert.png')

---
## 3. Seaborn — статистические графики
*(по мотивам `Visualization_Seaborn.ipynb`)*

Seaborn — надстройка над Matplotlib с красивым оформлением и богатыми статистическими возможностями.
Параметр `hue` добавляет разделение по категориям прямо на график.

In [ ]:
# histplot — распределение цен по районам с KDE
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

sns.histplot(
    data=krisha_df, x='price', hue='district',
    bins=30, kde=True, ax=axes[0], palette='Set1', alpha=0.6
)
axes[0].set_title('Распределение цен по районам (kde=True)', fontsize=13)
axes[0].set_xlabel('Цена (₸)')
axes[0].set_ylabel('Количество объявлений')

sns.histplot(
    data=krisha_df, x='Price_m2', y='district',
    bins=30, color='teal', ax=axes[1]
)
axes[1].set_title('Цена за м² × Район (2D гистограмма)', fontsize=13)
axes[1].set_xlabel('Цена за м² (₸)')
axes[1].set_ylabel('Район')

plt.tight_layout()
plt.savefig('plot_06_seaborn_histplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 6 сохранён: plot_06_seaborn_histplot.png')

In [ ]:
# boxplot — распределение цены за м² по типу дома и FloorType
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

# Boxplot 1: Price_m2 по типу дома и району
sns.boxplot(
    data=krisha_df, x='house_type', y='Price_m2',
    hue='district', ax=axes[0], palette='Set2',
    medianprops={'color': 'red', 'linewidth': 2}
)
axes[0].set_title('Цена за м² по типу дома и району', fontsize=13)
axes[0].set_xlabel('Тип дома')
axes[0].set_ylabel('Цена за м² (₸)')
axes[0].grid(True, alpha=0.3, axis='y')

# Boxplot 2: Возраст клиентов по оттоку (Churn)
churn_df['Статус'] = churn_df['Exited'].map({0: 'Остался', 1: 'Ушёл'})
sns.boxplot(
    data=churn_df, x='Geography', y='Age',
    hue='Статус', ax=axes[1], palette='coolwarm',
    medianprops={'color': 'black', 'linewidth': 2}
)
axes[1].set_title('Возраст клиентов по стране и оттоку', fontsize=13)
axes[1].set_xlabel('Страна')
axes[1].set_ylabel('Возраст')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plot_07_seaborn_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 7 сохранён: plot_07_seaborn_boxplot.png')

In [ ]:
# barplot — средний показатель по кварталам (аналог урока с COVID по кварталам)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

# Barplot 1: Средняя цена по кварталу и району (Krisha)
sns.barplot(
    data=krisha_df, x='Quarter', y='Price_m2',
    hue='district', ax=axes[0], palette='muted'
)
axes[0].set_title('Средняя цена за м² по кварталам и районам', fontsize=13)
axes[0].set_xlabel('Квартал')
axes[0].set_ylabel('Средняя цена за м² (₸)')
axes[0].grid(True, alpha=0.3, axis='y')

# Barplot 2: Доля оттока по числу продуктов (Churn)
churn_pivot = churn_df.groupby('NumOfProducts')['Exited'].mean().reset_index()
churn_pivot['Exited'] = churn_pivot['Exited'] * 100
sns.barplot(
    data=churn_pivot, x='NumOfProducts', y='Exited',
    ax=axes[1], palette='Reds_d'
)
axes[1].set_title('Доля оттока по числу продуктов банка (%)', fontsize=13)
axes[1].set_xlabel('Число продуктов')
axes[1].set_ylabel('Доля ушедших (%)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plot_08_seaborn_barplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 8 сохранён: plot_08_seaborn_barplot.png')

In [ ]:
# scatterplot — взаимосвязь двух числовых признаков
fig = plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(
    data=krisha_df,
    x='area', y='Price_m2',
    hue='district',
    size='floor_count',
    sizes=(20, 200),
    alpha=0.6,
    palette='Set1'
)
scatter.set_title('Взаимосвязь: площадь vs цена за м² (размер точки = этажность)', fontsize=13)
scatter.set_xlabel('Площадь (м²)')
scatter.set_ylabel('Цена за м² (₸)')
scatter.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_09_seaborn_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 9 сохранён: plot_09_seaborn_scatter.png')

In [ ]:
# heatmap — сводная тепловая карта: год постройки × район → средняя цена за м²
pivot_year = krisha_df.pivot_table(
    values='Price_m2',
    index='construction_year',
    columns='district',
    aggfunc='mean'
).round(0)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))

# Тепловая карта 1: Krisha — год постройки × район
sns.heatmap(
    pivot_year, annot=True, fmt='.0f',
    cmap='YlOrRd', linewidths=0.5,
    ax=axes[0], cbar_kws={'label': '₸/м²'}
)
axes[0].set_title('Средняя цена за м² (₸) по году постройки и району', fontsize=12)
axes[0].set_xlabel('Район')
axes[0].set_ylabel('Год постройки')

# Тепловая карта 2: Churn — корреляционная матрица числовых признаков
numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'Exited']
corr = churn_df[numeric_cols].corr()
mask = corr.abs() < 0.03
sns.heatmap(
    corr, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=axes[1],
    mask=mask,
    cbar_kws={'label': 'Корреляция'}
)
axes[1].set_title('Корреляционная матрица признаков оттока клиентов', fontsize=12)

plt.tight_layout()
plt.savefig('plot_10_seaborn_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 10 сохранён: plot_10_seaborn_heatmap.png')

---
## 4. Круговая диаграмма (Pie chart)
*(по мотивам `Types_of_visualization.ipynb` — раздел 5)*

Круговая диаграмма показывает **структуру** (вклад каждой категории в целое).
Используется для `character-deaths.csv` — датасет «Игры престолов».

In [ ]:
# Круговая диаграмма — смерти персонажей GoT по фракциям (топ-7)
deaths_count = got_df['Allegiances'].value_counts().head(7)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))

# Pie chart
wedge_props = {'edgecolor': 'white', 'linewidth': 2}
axes[0].pie(
    deaths_count.values,
    labels=deaths_count.index,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=wedge_props,
    colors=plt.cm.Set3.colors[:7]
)
axes[0].set_title('Смертность персонажей GoT по фракциям (топ-7)', fontsize=13)

# Альтернатива — горизонтальный barplot (более информативен)
axes[1].barh(deaths_count.index[::-1], deaths_count.values[::-1],
             color=plt.cm.Set3.colors[:7])
axes[1].set_title('То же, но столбчатая диаграмма (легче сравнивать)', fontsize=13)
axes[1].set_xlabel('Количество персонажей')
axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(deaths_count.values[::-1]):
    axes[1].text(v + 0.3, i, str(v), va='center')

plt.tight_layout()
plt.savefig('plot_11_pie_got.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 11 сохранён: plot_11_pie_got.png')

In [ ]:
# Смерти по книгам (столбчатая многоуровневая + круговая для гендера)
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 5))

# Смерти по книгам
book_deaths = got_df.dropna(subset=['Book of Death']).groupby(
    ['Book of Death', 'Nobility']
).size().unstack(fill_value=0)
book_deaths.index = [f'Книга {int(i)}' for i in book_deaths.index]
book_deaths.columns = ['Простолюдины', 'Знать']
book_deaths.plot(kind='bar', ax=axes[0], colormap='Accent', edgecolor='white')
axes[0].set_title('Смерти по книгам GoT (знать vs простолюдины)', fontsize=13)
axes[0].set_xlabel('Книга')
axes[0].set_ylabel('Число смертей')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

# Соотношение погибших по полу
gender_deaths = got_df['Gender'].map({0: 'Женщины', 1: 'Мужчины'}).value_counts()
axes[1].pie(
    gender_deaths.values,
    labels=gender_deaths.index,
    autopct='%1.1f%%',
    colors=['#FF6B9D', '#4A90D9'],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    startangle=90
)
axes[1].set_title('Соотношение погибших по полу (GoT)', fontsize=13)

plt.tight_layout()
plt.savefig('plot_12_got_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print('График 12 сохранён: plot_12_got_analysis.png')

---
## 5. Plotly Express — интерактивные графики
*(по мотивам `Visualizaion_Plotly.ipynb`)*

Plotly строит **интерактивные** графики: можно приближать, отдалять и смотреть точные значения при наведении.
Поддерживает: `line`, `bar`, `scatter`, `box`, `histogram`, `treemap`, `scatter_3d`.

In [ ]:
# Plotly line — динамика числа объявлений и средней цены по месяцам
monthly_plotly = krisha_df.groupby(['Month', 'district']).agg(
    listings=('price', 'count'),
    mean_price_m2=('Price_m2', 'mean')
).reset_index()
monthly_plotly['mean_price_m2'] = monthly_plotly['mean_price_m2'].round(0)

fig = px.line(
    data_frame=monthly_plotly,
    x='Month',
    y='mean_price_m2',
    color='district',
    markers=True,
    title='Динамика средней цены за м² по месяцам (Krisha.kz, Астана 2026)',
    labels={'Month': 'Месяц', 'mean_price_m2': 'Средняя цена за м² (₸)', 'district': 'Район'},
    height=450, width=900
)
fig.update_layout(hovermode='x unified')
fig.write_html('plot_13_plotly_line.html')
fig.show()
print('График 13 сохранён: plot_13_plotly_line.html')

In [ ]:
# Plotly bar — топ-10 ЖК по средней цене за м²
top_complex = (
    krisha_df.groupby(['complex_name', 'district'])['Price_m2']
    .agg(['mean', 'count'])
    .query('count > 10')
    .reset_index()
    .sort_values('mean', ascending=False)
    .head(15)
)
top_complex.columns = ['ЖК', 'Район', 'Средняя цена за м²', 'Объявлений']
top_complex['Средняя цена за м²'] = top_complex['Средняя цена за м²'].round(0)

fig = px.bar(
    data_frame=top_complex,
    x='ЖК',
    y='Средняя цена за м²',
    color='Район',
    text='Средняя цена за м²',
    title='Топ-15 ЖК по средней цене за м² (Астана, Есиль и Нура)',
    height=500, width=1000,
    color_discrete_map={'Есильский р-н': '#1976D2', 'Нура р-н': '#F57C00'}
)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-40)
fig.write_html('plot_14_plotly_bar_complex.html')
fig.show()
print('График 14 сохранён: plot_14_plotly_bar_complex.html')

In [ ]:
# Plotly scatter — взаимосвязь площади и цены (размер точки = этажность, цвет = район)
fig = px.scatter(
    data_frame=krisha_df,
    x='area',
    y='price',
    color='district',
    size='floor_count',
    hover_data=['complex_name', 'construction_year', 'Price_m2', 'house_type'],
    title='Площадь vs Цена квартиры (Астана 2026) — интерактивная диаграмма рассеяния',
    labels={'area': 'Площадь (м²)', 'price': 'Цена (₸)', 'district': 'Район'},
    color_discrete_map={'Есильский р-н': '#1976D2', 'Нура р-н': '#F57C00'},
    height=550, width=1000
)
fig.write_html('plot_15_plotly_scatter.html')
fig.show()
print('График 15 сохранён: plot_15_plotly_scatter.html')

In [ ]:
# Plotly treemap — иерархическая диаграмма: Район → Тип дома → ЖК
treemap_data = (
    krisha_df.groupby(['district', 'house_type', 'complex_name'])
    .agg(count=('price', 'count'), mean_pm2=('Price_m2', 'mean'))
    .reset_index()
    .query('count > 3')
)
treemap_data['mean_pm2'] = treemap_data['mean_pm2'].round(0)

fig = px.treemap(
    data_frame=treemap_data,
    path=['district', 'house_type', 'complex_name'],
    values='count',
    color='mean_pm2',
    color_continuous_scale='RdYlGn_r',
    title='Иерархическая карта рынка недвижимости Астаны (размер = число объявлений, цвет = цена за м²)',
    height=600, width=1000
)
fig.write_html('plot_16_plotly_treemap.html')
fig.show()
print('График 16 сохранён: plot_16_plotly_treemap.html')

In [ ]:
# Plotly box — интерактивный boxplot оттока клиентов по возрасту и стране
fig = px.box(
    data_frame=churn_df,
    x='Geography',
    y='Age',
    color='Статус',
    points='outliers',
    title='Распределение возраста клиентов по стране и статусу оттока',
    labels={'Geography': 'Страна', 'Age': 'Возраст', 'Статус': 'Статус клиента'},
    height=500, width=900,
    color_discrete_map={'Остался': '#4CAF50', 'Ушёл': '#F44336'}
)
fig.write_html('plot_17_plotly_box_churn.html')
fig.show()
print('График 17 сохранён: plot_17_plotly_box_churn.html')

In [ ]:
# Plotly scatter_3d — 3D диаграмма рассеяния: цена, площадь, этаж по году постройки
fig = px.scatter_3d(
    data_frame=krisha_df,
    x='area',
    y='Price_m2',
    z='floor',
    color='district',
    symbol='house_type',
    hover_data=['complex_name', 'construction_year'],
    title='3D: Площадь × Цена за м² × Этаж (Астана 2026)',
    labels={'area': 'Площадь (м²)', 'Price_m2': 'Цена за м² (₸)', 'floor': 'Этаж'},
    color_discrete_map={'Есильский р-н': '#1976D2', 'Нура р-н': '#F57C00'},
    height=700, width=1000
)
fig.write_html('plot_18_plotly_scatter3d.html')
fig.show()
print('График 18 сохранён: plot_18_plotly_scatter3d.html')

---
## Итоговая таблица использованных типов визуализации

| № | Тип графика | Библиотека | Датасет | Цель |
|---|---|---|---|---|
| 1 | Линейный (`line`) | Pandas | Krisha | Динамика объявлений по месяцам |
| 2 | Гистограмма (`hist`) | Pandas | Krisha | Распределение цен за м² |
| 3 | Столбчатая (`bar`) | Pandas | Krisha | Сравнение районов по типу дома |
| 4 | 3 графика рядом | Matplotlib | Churn | Анализ оттока клиентов |
| 5 | Scatter + врезка | Matplotlib | Krisha | Площадь vs Цена (insert axes) |
| 6 | Гистограмма (`histplot`) | Seaborn | Krisha | Распределение цен с KDE |
| 7 | Ящик с усами (`boxplot`) | Seaborn | Krisha+Churn | Сравнение распределений |
| 8 | Столбчатая (`barplot`) | Seaborn | Krisha+Churn | Средние по кварталам/продуктам |
| 9 | Рассеяния (`scatterplot`) | Seaborn | Krisha | Взаимосвязь площади и цены за м² |
| 10 | Тепловая карта (`heatmap`) | Seaborn | Krisha+Churn | Сводные таблицы и корреляции |
| 11 | Круговая + barh | Matplotlib | GoT | Структура смертей по фракциям |
| 12 | Bar + Pie | Matplotlib | GoT | Смерти по книгам и полу |
| 13 | Линейный интерактивный | Plotly | Krisha | Динамика цены по месяцам |
| 14 | Столбчатый интерактивный | Plotly | Krisha | Топ-15 ЖК по цене за м² |
| 15 | Scatter интерактивный | Plotly | Krisha | Площадь vs Цена (hover) |
| 16 | Treemap (иерархическая) | Plotly | Krisha | Структура рынка: район → тип → ЖК |
| 17 | Box интерактивный | Plotly | Churn | Возраст клиентов по оттоку |
| 18 | 3D Scatter | Plotly | Krisha | Площадь × Цена × Этаж |

**Итого: 18 графиков, все 4 библиотеки, все 3 датасета.**